# M00: Setup, images as arrays, class distribution

> Academic research prototype; outputs are not a medical diagnosis.

Checks the environment, loads the APTOS 2019 labels, shows one image per grade, splits an image into R/G/B channels and plots the class distribution. **Not executed in the repository; run it on Kaggle or locally with the data present.**

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd()
for cand in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (cand / "src" / "drscreen").is_dir():
        sys.path.insert(0, str(cand / "src")); ROOT = cand; break
# On Kaggle the package is embedded in the kernel script; for notebooks, upload src/drscreen as a Dataset or copy it next to the notebook.
DATA_ROOT = os.environ.get("DR_DATA_ROOT") or ("/kaggle/input/aptos2019-blindness-detection" if Path("/kaggle/input").exists() else str(ROOT / "data" / "aptos2019"))
print("project root:", ROOT, "| data root:", DATA_ROOT)

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
from drscreen.datasets import load_labelled_dataset
from drscreen.preprocess import read_image
df = load_labelled_dataset(DATA_ROOT, 'APTOS 2019')
print(len(df), 'labelled images; dropped:', df.attrs['dropped'])
print(df['grade'].value_counts().sort_index())
print('referable prevalence (grade>=2): %.3f' % (df['grade'] >= 2).mean())

In [ ]:
rng = np.random.default_rng(0)
sample = df.sample(10, random_state=0)
for r in sample.itertuples():
    im = read_image(r.path)
    print(r.image_id, 'grade', r.grade, 'shape', im.shape, 'dtype', im.dtype, 'min/max', im.min(), im.max())

In [ ]:
from drscreen.metrics import GRADE_NAMES
fig, axs = plt.subplots(2, 5, figsize=(15, 6))
for g in range(5):
    r = df[df.grade == g].iloc[0]
    im = read_image(r.path)
    axs[0, g].imshow(im); axs[0, g].set_title(GRADE_NAMES[g]); axs[0, g].axis('off')
    axs[1, g].imshow(im[..., 1], cmap='gray'); axs[1, g].set_title('green channel'); axs[1, g].axis('off')
plt.tight_layout(); plt.show()
# Test: the green channel shows vessels and red lesions with the highest contrast (haemoglobin absorbs green light).

In [ ]:
ax = df['grade'].value_counts().sort_index().plot.bar(figsize=(5, 3), title='APTOS 2019: images per grade'); ax.set_xlabel('ICDR grade'); plt.show()